# 04 - Memory: short-term sessions vs long-term memory

**Maple Mortgage Assistant, Northwind Home Lending (fictional)**

## What you'll learn
- **Short-term memory**: `AgentSession` keeps the turns of one conversation; it can be saved and restored
- **Long-term memory**: a Foundry **memory store** extracts durable facts (preferences) and recalls them in a *new* session
- **Scope** isolates memories per borrower; `delete_scope` implements right-to-erasure
- FSI notes: what you should and should not store

> **[preview]** Foundry memory stores are a preview feature (`allow_preview=True`). If your tenant/region lacks it, the create cell prints the real error.

In [1]:
import os, sys, asyncio, json
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

DAY2 = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(DAY2 / ".env", override=False)
sys.path.insert(0, str(DAY2))

REQUIRED = ["AZURE_AI_PROJECT_ENDPOINT", "AZURE_AI_MODEL_DEPLOYMENT_NAME", "AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"]
for name in REQUIRED:
    print(f"{name:42s}", "set" if os.environ.get(name) else "MISSING")   # never print values
assert all(os.environ.get(n) for n in REQUIRED), "Run scripts/write-env.sh first"

credential = DefaultAzureCredential()

import time
from agent_framework import Agent, AgentSession
from agent_framework.foundry import FoundryChatClient, FoundryMemoryProvider
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MemoryStoreDefaultDefinition, MemoryStoreDefaultOptions

ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
CHAT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
EMBED = os.environ["AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"]
STORE = "maple-borrower-memory"

client = FoundryChatClient(project_endpoint=ENDPOINT, model=CHAT, credential=credential)
pc = AIProjectClient(endpoint=ENDPOINT, credential=credential, allow_preview=True)   # memory stores are preview
from azure.ai.projects.aio import AIProjectClient as AsyncProjectClient
from azure.identity.aio import DefaultAzureCredential as AsyncCredential
pc_async = AsyncProjectClient(endpoint=ENDPOINT, credential=AsyncCredential(), allow_preview=True)   # the AF provider awaits its client
INSTRUCTIONS = ("You are Maple, the mortgage assistant for Northwind Home Lending (fictional). Never promise a rate or approval. "
                "Use what you remember about the borrower; if you do not remember, say so.")
print("ready")

AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set


ready


## 1. Short-term memory: one session
The session carries the conversation. A brand-new session starts empty.

In [2]:
agent = Agent(client=client, name="maple-short", instructions=INSTRUCTIONS)

s1 = agent.create_session()
r = await agent.run("Hi, I'm Priya. I'm a first-time buyer looking at condos around 450k.", session=s1)
print("turn 1:", r.text[:200], "\n")
r = await agent.run("What price range did I mention?", session=s1)
print("same session  :", r.text[:200], "\n")

s2 = agent.create_session()
r = await agent.run("What price range did I mention?", session=s2)
print("new session   :", r.text[:200])

turn 1: Hi Priya! Great to hear you're looking at condos around $450,000. As a first-time buyer, I can help guide you through the mortgage options and process. To get started, could you share a bit about your 



same session  : You mentioned you’re looking at condos around $450,000. How can I assist you further with your home buying journey? 



new session   : I don’t have a record of the price range you mentioned. Could you please remind me of the details?


### Save and restore a session
`to_dict()` gives JSON you can store (Cosmos, Redis, a file). Restoring it resumes the same conversation.

In [3]:
saved = json.dumps(s1.to_dict())
print("serialized session:", len(saved), "bytes")
restored = AgentSession.from_dict(json.loads(saved))
r = await agent.run("Remind me: what was my name and price range?", session=restored)
print("restored session:", r.text[:200])

serialized session: 167 bytes


restored session: Your name is Priya, and you’re looking at condos around $450,000. Let me know how I can assist you next!


Short-term memory is the *conversation*. It is lost when the session object is dropped and grows with every turn.
For things that must survive across conversations (preferences, goals) we need long-term memory.

## 2. Create a long-term memory store [preview]
`user_profile_enabled` extracts stable facts about the user; `chat_summary_enabled` keeps conversation summaries.

In [4]:
definition = MemoryStoreDefaultDefinition(
    chat_model=CHAT, embedding_model=EMBED,
    options=MemoryStoreDefaultOptions(user_profile_enabled=True, chat_summary_enabled=True),
)
try:
    pc.beta.memory_stores.get(STORE)
    print("store already exists:", STORE)
except Exception:
    store = pc.beta.memory_stores.create(name=STORE, definition=definition,
                                         description="Borrower preferences for Maple (fictional data only)")
    print("created store:", store.name)

store already exists: maple-borrower-memory


## 3. Agent with `FoundryMemoryProvider`
`scope` is the isolation key. Use an **opaque customer id** (never name, SIN/SSN, email).
`update_delay=0` writes memories right after each run (default is 300 s, which batches updates; fine for production, slow for a demo).

In [5]:
def maple_with_memory(scope: str) -> Agent:
    provider = FoundryMemoryProvider(project_client=pc_async, memory_store_name=STORE, scope=scope, update_delay=0)
    return Agent(client=client, name="maple-memory", instructions=INSTRUCTIONS, context_providers=[provider])

PRIYA = "cust-demo-001"     # opaque id, not a name
agent_p = maple_with_memory(PRIYA)

# Session A: Priya states preferences
sess_a = agent_p.create_session()
r = await agent_p.run(
    "I prefer a 5 year fixed term, I want bi-weekly payments, and I would rather not use a variable rate. "
    "I'm a first-time buyer and my budget is around 450k.", session=sess_a)
print("session A:", r.text[:300])

session A: Thanks for sharing your preferences! A 5-year fixed term with bi-weekly payments can provide predictability in your mortgage budget, which is great for a first-time buyer. Since you prefer to avoid variable rates, focusing on fixed-rate options would be the best fit.

With a budget around $450,000, 


Extraction runs server side. Poll the store until the facts appear (usually seconds).

In [6]:
from azure.ai.projects.models import MemorySearchOptions

def recall(scope, query, tries=1):
    for i in range(tries):
        res = pc.beta.memory_stores.search_memories(STORE, scope=scope, items=query, options=MemorySearchOptions(max_memories=5))
        mems = [m.as_dict() for m in res.memories]
        if mems or i == tries - 1:
            return mems
        time.sleep(6)

mems = recall(PRIYA, "borrower preferences", tries=12)
print(f"{len(mems)} memories for scope {PRIYA}:")
for m in mems:
    item = m.get("memory_item", m)
    print(" -", item.get("kind", ""), "|", item.get("content", item))

5 memories for scope cust-demo-001:
 - chat_summary | On November 16, 2026, the user indicated preferences for a mortgage loan: a 5-year fixed term, bi-weekly payments, a preference against variable rates, and that they are a first-time buyer with a budget around $450,000. The assistant acknowledged these preferences and offered to help guide the user through mortgage product options aligned with their criteria, requesting additional details such as credit score, employment status, and down payment amount for more precise recommendations.
 - user_profile | User prefers mortgage options avoiding variable rates.
 - user_profile | User prefers bi-weekly mortgage payments.
 - user_profile | User prefers a 5-year fixed mortgage term.
 - user_profile | User is a first-time home buyer.


## 4. A new session weeks later
No history is passed: the provider retrieves the facts from the store.

In [7]:
sess_b = agent_p.create_session()          # brand new session, empty history
r = await agent_p.run("I'm back. What term and payment frequency did I prefer, and what was my budget?", session=sess_b)
print("session B:", r.text[:400])

session B: Welcome back! You preferred a 5-year fixed mortgage term with bi-weekly payments. Your home purchase budget is around $450,000. Let me know if you'd like to discuss mortgage options based on these preferences!


## 5. Isolation between borrowers
A second customer asks the same question under a different scope.

In [8]:
OTHER = "cust-demo-002"
agent_o = maple_with_memory(OTHER)
r = await agent_o.run("What term and payment frequency did I prefer?", session=agent_o.create_session())
print("other borrower:", r.text[:300])
print("memories in other scope:", len(recall(OTHER, "borrower preferences")))

other borrower: I don't recall your preferred loan term or payment frequency. Could you please provide those details again?


memories in other scope: 0


## 6. Right to erasure
`delete_scope` removes everything stored for one borrower. Verify with a search afterwards.

In [9]:
res = pc.beta.memory_stores.delete_scope(STORE, scope=PRIYA)
print("delete_scope ->", res.as_dict() if hasattr(res, "as_dict") else res)
print("memories left for", PRIYA, ":", len(recall(PRIYA, "borrower preferences")))
r = await agent_p.run("What term did I prefer?", session=agent_p.create_session())
print("after erasure :", r.text[:300])

delete_scope -> {'object': 'memory_store.scope.deleted', 'name': 'maple-borrower-memory', 'scope': 'cust-demo-001', 'deleted': True}


memories left for cust-demo-001 : 0


after erasure : I don't have a record of your preferred loan term right now. Could you please remind me what you were considering?


## FSI notes: PII and retention
| Topic | Guidance |
|---|---|
| What to store | Preferences and goals (term, payment frequency, budget band). **Not** SIN/SSN, full account numbers, health or credit-bureau data. |
| Key | Opaque customer id as `scope`; map to the person in your CRM, not in the memory store. |
| Redaction | Mask PII before it reaches the model/memory (guardrail or pre-processing); the store keeps what the model saw. |
| Retention | Define a TTL per policy (e.g. delete scope at account closure or after N months of inactivity) and run `delete_scope` from a scheduled job. |
| Erasure | `delete_scope` covers right-to-erasure requests; log the request id, not the content. |
| Audit | Memory is *derived* data and can be wrong; do not use it for credit decisions. Keep decisions based on verified application data. |
| Access | Entra ID only; the caller needs Azure AI User on the project. Keep the store in the same region/residency boundary as the project. |

## Key takeaways
- Session = this conversation; memory store = facts across conversations.
- `scope` is your tenant/borrower boundary: test isolation explicitly.
- Plan deletion (retention + erasure) before you plan recall.
- Memory stores are preview: pin versions and keep a fallback (your own store of preferences).

**Next up:** [05 - Skills and registry](05-skills-and-registry.ipynb)